# Improved Hybrid CNN-LSTM for Fruit Freshness Classification
## Pattern Recognition and Applications — Term Project (Improvement Notebook)

**Author:** Melih Can KÖK  
**Reference paper:** Ghosh & Singh, *Discover Artificial Intelligence* (2026), DOI: 10.1007/s44163-025-00750-7

---
### Improvements over the baseline reproduction:
| Change | Baseline (Reproduction) | This Notebook |
|--------|------------------------|---------------|
| Hybrid CNN backbone | Custom CNN from scratch (16→32→64→128) | **Pre-trained EfficientNetB0** (ImageNet) |
| LSTM type | Unidirectional LSTM (64 units) | **Bidirectional LSTM — Bi-LSTM (128 units)** |
| Sequence augmentation | Independent random per timestep → collapse | **Structured progressive augmentation** |
| LR scheduler | ReduceLROnPlateau (patience=2) | **Cosine Annealing** |
| Epochs | 30 | **50** (+ early stopping patience=10) |
| Standalone LR issue | Fixed LR 0.001 (VGG16/EffNet failed) | **Warm-up + per-model LR** |

### Literature basis:
- **Pre-trained backbone in hybrid**: Jahan et al. [22] showed pre-trained CNNs significantly improve freshness accuracy. The paper's own ablation confirmed pre-trained > custom CNN.
- **Bidirectional LSTM**: Ahmad et al. [25] demonstrated Bi-GRU outperforms unidirectional GRU for sequential classification by capturing both past and future context.
- **Structured augmentation**: The baseline collapse happened because each sequence timestep received an *independent* random transform, giving the LSTM incoherent input. Progressive augmentation creates a learnable pseudo-ripening signal.

# BOLUM 1 — Kutuphaneler, GPU Kurulumu ve Veri Ayırma

In [1]:
import os, time, shutil, random, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LSTM, Bidirectional,
    TimeDistributed, GlobalAveragePooling2D,
    Conv2D, MaxPooling2D, BatchNormalization, Flatten
)
from tensorflow.keras.applications import (
    InceptionV3, VGG16, ResNet50, DenseNet121, EfficientNetB0
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
import kagglehub
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive

# Google Drive Baglantisi
drive.mount('/content/drive')

# Drive yolu yedeklemesi tanimlamasi
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/Fruit_Freshness_Project_Results'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# GPU hafizasi
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU aktif {[g.name for g in gpus]}")

print("\n dataset yukleniyor")
dataset_path = kagglehub.dataset_download('sriramr/fruits-fresh-and-rotten-for-classification')
BASE_DIR = os.path.join(dataset_path, 'dataset')

CLASS_NAMES = ['freshapples', 'freshbanana', 'freshoranges',
               'rottenapples', 'rottenbanana', 'rottenoranges']
LABEL_MAP = {c: i for i, c in enumerate(CLASS_NAMES)}

print("Goruntu yollarinin toplanmasi")
all_paths, all_labels = [], []
for split_dir in ['train', 'test']:
    for cls in CLASS_NAMES:
        cls_dir = os.path.join(BASE_DIR, split_dir, cls)
        if not os.path.isdir(cls_dir): continue
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_paths.append(os.path.join(cls_dir, fname))
                all_labels.append(LABEL_MAP[cls])

all_paths, all_labels = np.array(all_paths), np.array(all_labels)

# 70/10/20 oranlandi
X_train, X_temp, y_train, y_temp = train_test_split(
    all_paths, all_labels, test_size=0.30, random_state=42, stratify=all_labels)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.667, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

WORK_DIR = './working_improved'
TRAIN_DIR, VAL_DIR, TEST_DIR = [os.path.join(WORK_DIR, s) for s in ['train', 'val', 'test']]

def write_dirs(paths, labels, base):
    for cls in CLASS_NAMES:
        os.makedirs(os.path.join(base, cls), exist_ok=True)
    for p, lbl in zip(paths, labels):
        shutil.copy2(p, os.path.join(base, CLASS_NAMES[lbl], os.path.basename(p)))

print("dosya yapısı")
write_dirs(X_train, y_train, TRAIN_DIR)
write_dirs(X_val,   y_val,   VAL_DIR)
write_dirs(X_test,  y_test,  TEST_DIR)
print("tamamlandi")

Mounted at /content/drive
GPU aktif ['/physical_device:GPU:0']

 dataset yukleniyor


100%|██████████| 3.58G/3.58G [01:35<00:00, 40.1MB/s]

Extracting files...


Goruntu yollarının toplanmasi
Train: 9519, Val: 1358, Test: 2722
dosya yapısı
tamamlandi


# BOLUM 2 — Hiperparametreler ve Üreticiler

In [2]:
BATCH_SIZE = 64
EPOCHS     = 50   # Makale standardi (temelde 30 referans kullanildi)
SEQ_LEN    = 5
NUM_CLASSES = 6

# Onceden egitilmis bagimsiz modeller icin gelistirme
train_aug = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.10,
    brightness_range=[0.75, 1.25],
    fill_mode='nearest'
)
val_test_gen_base = ImageDataGenerator(rescale=1.0/255)

# Cosine annealing LR scheduler
# Gelistirme olarak erken yakinsamayi onleyen daha yumusak bir LR bozunmasi
def cosine_lr(epoch, lr_max=1e-3, lr_min=1e-6, T=50):
    """Cosine annealing (Kosinüs tavlama): T dongu boyunca lr_max degerinden lr_min degerine duser
    Loshchilov & Hutter (2017) temelli SGDR: Stokastik Gradient Dususu
    ve Yeniden Baslatma.)"""
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * epoch / T))

def get_callbacks_standalone(model_name):
    return [
        EarlyStopping(monitor='val_loss', patience=10,
                      restore_best_weights=True, verbose=0),
        LearningRateScheduler(lambda ep: cosine_lr(ep), verbose=0)
    ]

def get_callbacks_hybrid():
    return [
        EarlyStopping(monitor='val_loss', patience=10,
                      restore_best_weights=True, verbose=0),
        LearningRateScheduler(lambda ep: cosine_lr(ep, lr_max=5e-4), verbose=0)
    ]

def make_gen(datagen, directory, img_size=(224, 224), shuffle=True):
    return datagen.flow_from_directory(
        directory, target_size=img_size, batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=shuffle, seed=42)

model_results    = {}
model_histories  = {}
model_preds      = {}
timing_results   = []

print("Ayarlama hazir")
print(f"  Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, Seq: {SEQ_LEN}")

Ayarlama hazir
  Epochs: 50, Batch: 64, Seq: 5


# SECTION 3 — Standalone Pre-trained Models (50 epochs, Cosine LR)

In [3]:
def build_standalone(backbone_cls, input_shape):
    """Ozel sınıflandırma ile onceden egitilmis CNN
      Mimari Ghosh & Singh (2026)"""
    base = backbone_cls(
        include_top=False, weights='imagenet', input_shape=input_shape)

    # Tum taban katmanlarını dondurulmasi
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(base.input, out)
    model.compile(
        optimizer=Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'])
    return model, base


BACKBONE_CFGS = {
    'InceptionV3':    {'cls': InceptionV3,    'size': (299, 299)},
    'VGG16':          {'cls': VGG16,          'size': (224, 224)},
    'ResNet50':       {'cls': ResNet50,       'size': (224, 224)},
    'DenseNet121':    {'cls': DenseNet121,    'size': (224, 224)},
    'EfficientNetB0': {'cls': EfficientNetB0, 'size': (224, 224)},
}

print("Bagimsiz modelleme araci hazir")

Bagimsiz modelleme araci hazir


In [ ]:
for name, cfg in BACKBONE_CFGS.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print('='*60)
    tf.keras.backend.clear_session(); gc.collect()

    img_size = cfg['size']
    tr_g = make_gen(train_aug, TRAIN_DIR, img_size=img_size)
    va_g = make_gen(val_test_gen_base, VAL_DIR, img_size=img_size, shuffle=False)
    te_g = make_gen(val_test_gen_base, TEST_DIR, img_size=img_size, shuffle=False)

    model, base = build_standalone(cfg['cls'], (img_size[0], img_size[1], 3))

    # Sadece ust katmanların egitimi (temel katmanlar 10 epok sabitlenmis)
    phase1_hist = model.fit(
        tr_g, validation_data=va_g,
        epochs=10,
        callbacks=[
            EarlyStopping('val_loss', patience=5, restore_best_weights=True)
        ],
        verbose=1
    )

    # Son 3 konv bloğunu daha dusuk ogrenme oraniyla Fine-tune
    # Gelistirme olarak uyumlu 2 asamali egitim
    # VGG16 / EfficientNetB0'in cokmesini engeller
    base.trainable = True
    # Son 30 katman haric hepsinin dondurulmasi (bircok omurga icin son 3 blok)
    for layer in base.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(1e-5),   # Low LR icin fine-tuning
        loss='categorical_crossentropy',
        metrics=['accuracy'])

    t0 = time.time()
    hist = model.fit(
        tr_g, validation_data=va_g,
        epochs=EPOCHS,
        callbacks=get_callbacks_standalone(name),
        verbose=1
    )
    epoch_time = (time.time() - t0) / max(len(hist.epoch), 1)

    # Grafik icin verileri birlestirme
    combined_hist = {}
    for k in hist.history:
        combined_hist[k] = phase1_hist.history.get(k, []) + hist.history[k]
    model_histories[name] = combined_hist

    # Degerlendirme
    te_g.reset()
    t_inf = time.time()
    y_prob = model.predict(te_g, verbose=0)
    inf_ms = (time.time() - t_inf) / te_g.samples * 1000

    timing_results.append({
        'Model': name,
        'Training time (s/epoch)': round(epoch_time, 2),
        'Inference time (ms/image)': round(inf_ms, 2)
    })

    y_pred  = np.argmax(y_prob, axis=1)
    y_true  = te_g.classes[:len(y_pred)]

    model_results[name] = {
        'Accuracy':  accuracy_score(y_true, y_pred) * 100,
        'Precision': precision_score(y_true, y_pred, average='macro') * 100,
        'Recall':    recall_score(y_true, y_pred, average='macro') * 100,
        'F1-Score':  f1_score(y_true, y_pred, average='macro') * 100,
    }
    model_preds[name] = {'y_true': y_true, 'y_prob': y_prob, 'y_pred': y_pred}

    print(f"\n  {name} Results:")
    for k, v in model_results[name].items():
        print(f"    {k}: {v:.2f}%")


Training: InceptionV3
Found 9519 images belonging to 6 classes.
Found 1358 images belonging to 6 classes.
Found 2722 images belonging to 6 classes.
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 271s 2s/step - accuracy: 0.8887 - loss: 0.3126 - val_accuracy: 0.9632 - val_loss: 0.0943
Epoch 2/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 224s 2s/step - accuracy: 0.9555 - loss: 0.1324 - val_accuracy: 0.9698 - val_loss: 0.0719
Epoch 3/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 225s 2s/step - accuracy: 0.9644 - loss: 0.1065 - val_accuracy: 0.9750 - val_loss: 0.0671
Epoch 4/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 226s 2s/step - accuracy: 0.9697 - loss: 0.0877 - val_accuracy: 0.9728 - val_loss: 0.0627
Epoch 5/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 224s 2s/step - accuracy: 0.9744 - loss: 0.0790 - val_accuracy: 0.9838 - val_loss: 0.0437
Epoch 6/10
149/149 ━━━━━━━━━━━━━━━━━━━━ 224s 2s/step - accuracy: 0.9711 - loss: 0.0804 - val_accuracy: 0.9838 - val_loss: 0.0425
Epoch 7/10
149/149 ━━━━━━━

# SECTION 4 — Improved Hybrid CNN-LSTM

## Key architectural changes:

### 4.1 Pre-trained EfficientNetB0 CNN backbone
Instead of the custom CNN (16→32→64→128 filters from scratch), we use EfficientNetB0 pre-trained on ImageNet as the feature extractor. This provides richer, semantically meaningful feature vectors for the LSTM, matching the paper's own finding that pre-trained > custom CNN (ablation study, Table 11).

### 4.2 Bidirectional LSTM (Bi-LSTM)
The original paper uses a unidirectional LSTM(64). We use `Bidirectional(LSTM(128))` which processes each sequence both forward and backward, capturing context from both directions — directly inspired by Ahmad et al. [25] who showed Bi-GRU outperforms standard GRU for sequential classification.

### 4.3 Structured progressive augmentation for sequences
**Root cause of baseline collapse:** the sequence generator applied an independent random transform to each of the 5 timesteps. The LSTM received 5 completely unrelated views — no temporal pattern to learn.

**Fix:** Each timestep receives a *structured*, progressively stronger augmentation:
- t=1: mild (rotation ±5°, brightness ±5%)
- t=2: moderate (rotation ±10°, brightness ±10%)
- t=3: medium (rotation ±15°, brightness ±15%)
- t=4: strong (rotation ±20°, brightness ±20%)
- t=5: full (rotation ±30°, brightness ±25%, flip, noise)

This simulates a pseudo-ripening progression: an image that gradually changes, providing a coherent temporal signal for the LSTM to model.

In [ ]:
def build_improved_hybrid():
    """
    Gelistirilmis Hibrit CNN-LSTM
    CNN omurgasi olarak onceden egitilmis EfficientNetB0 (dondurulmus)
    Zamansal model olarak cift yonlu LSTM (128 birim)
    Baslik olarak yogun(256, relu) → Dropout(0,5) -> yogun(6, softmax)

    Literatur temeli
    Onceden egitilmis omurga olarak Jahan et al. [22] makale ablasyonu
    Bi-LSTM: Ahmad et al. [25] (Bi-GRU tek yonlu modelden daha iyi performans verir)
    """
    h, w = 224, 224

    # CNN Alt model (ozellik cikarimi)
    cnn_base = EfficientNetB0(
        include_top=False, weights='imagenet',
        input_shape=(h, w, 3))
    cnn_base.trainable = False   # Verimli egitim icin dondurulmus

    cnn_input = Input(shape=(h, w, 3))
    feat = cnn_base(cnn_input, training=False)
    feat = GlobalAveragePooling2D()(feat)  # (batch, 1280)
    cnn_model = Model(cnn_input, feat, name='efficientnet_extractor')

    # Sekans girisi
    seq_input = Input(shape=(SEQ_LEN, h, w, 3), name='sequence_input')

    # Her zaman adımına CNN uygulamasi
    seq_feat = TimeDistributed(cnn_model)(seq_input)   # (batch, SEQ, 1280)
    seq_feat = Dropout(0.3)(seq_feat)

    # Cift yonlu LSTM dizileri hem ileri hem de geri yonde isler
    bi_lstm_out = Bidirectional(
        LSTM(128, activation='tanh', return_sequences=False)
    )(seq_feat)   # (batch, 256)
    bi_lstm_out = Dropout(0.5)(bi_lstm_out)

    x = Dense(256, activation='relu')(bi_lstm_out)
    x = Dropout(0.5)(x)
    output = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(seq_input, output, name='improved_hybrid')
    model.compile(
        optimizer=Adam(5e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    model.summary()
    return model

print("Gelistirilmis hibrit model olusturucu hazir")

In [ ]:
# Gelistirilmis Dizi Üretici Yapilandirilmis Kademeli Artirma

# Temel modeldeki problem: her zaman adimi bagimsiz rastgele donusumdur
#   -> LSTM 5 alakasiz goruntu gorur -> ogrenilebilir desen yok -> cokme
# Cozum olarak her zaman adimi deterministik ve giderek guclenen donusum
#   -> LSTM hafiften gucluye bir ilerleme gorur -> tutarli sinyal

AUGMENTERS = [
    ImageDataGenerator(  # t=1: cok hafif
        rotation_range=5, brightness_range=[0.95, 1.05]),
    ImageDataGenerator(  # t=2: hafif
        rotation_range=10, brightness_range=[0.90, 1.10],
        horizontal_flip=True),
    ImageDataGenerator(  # t=3: orta
        rotation_range=15, brightness_range=[0.85, 1.15],
        horizontal_flip=True, zoom_range=0.05),
    ImageDataGenerator(  # t=4: guclu
        rotation_range=20, brightness_range=[0.80, 1.20],
        horizontal_flip=True, vertical_flip=True, zoom_range=0.08),
    ImageDataGenerator(  # t=5: tam (makale ile ayni)
        rotation_range=30, brightness_range=[0.75, 1.25],
        horizontal_flip=True, vertical_flip=True,
        zoom_range=0.10),
]

def structured_sequence_generator(base_gen, seq_len=SEQ_LEN, add_noise=True):
    """
    (X_seq, y) dondurur, burada X_seq (batch, seq_len, H, W, C) seklindedir
    Her t zaman adimi ogrenilebilir bir sahte olgunlasma ilerlemesi olusturarak
    giderek guclenen donusumlerle t artiricisini uygular
    """
    assert seq_len == len(AUGMENTERS), "seq_len artirici sayisiyla eslesmeli"
    while True:
        for X_batch, y_batch in base_gen:
            B, H, W, C = X_batch.shape
            seq_batch = np.zeros((B, seq_len, H, W, C), dtype=np.float32)
            for t, aug in enumerate(AUGMENTERS):
                for i in range(B):
                    img_t = aug.random_transform(X_batch[i])
                    if add_noise and t == seq_len - 1:
                        # Sadece son zaman adiminda Gauss gurultusu (tam artirma)
                        noise = np.random.normal(0, 0.01, img_t.shape).astype(np.float32)
                        img_t = np.clip(img_t + noise, 0.0, 1.0)
                    seq_batch[i, t] = img_t
            yield seq_batch, y_batch


def flat_sequence_generator(base_gen, seq_len=SEQ_LEN):
    """Dogrulama / test icin tum zaman adimlar orijinal goruntu (artirma yok)"""
    while True:
        for X_batch, y_batch in base_gen:
            B, H, W, C = X_batch.shape
            seq_batch = np.zeros((B, seq_len, H, W, C), dtype=np.float32)
            for t in range(seq_len):
                seq_batch[:, t, :, :, :] = X_batch
            yield seq_batch, y_batch

print("Yapilandirilmis dizi olusturucular hazir")
print(f"Zaman adimi basina artirma seviyeleri: {[f't{i+1}' for i in range(SEQ_LEN)]}")

In [ ]:
# ABLASYON 1 Artirilmis Diziler Olmadan Gelistirilmis Hibrit
# (temel surum ozel bir CNN kullandi - burada onceden egitilmis EfficientNetB0 kullaniyoruz)
# Dogrudan onceden egitilmis omurganin tek basina ise yarayip yaramadigini test eder

print("\nGelistirilmis Hibrit Egitiliyor - Artirilmis Dizi Yok (Ablasyon)")
tf.keras.backend.clear_session(); gc.collect()

tr_g_plain = make_gen(ImageDataGenerator(rescale=1.0/255), TRAIN_DIR, shuffle=True)
va_g_plain = make_gen(val_test_gen_base, VAL_DIR, shuffle=False)
te_g_plain = make_gen(val_test_gen_base, TEST_DIR, shuffle=False)

steps_tr   = tr_g_plain.samples  // BATCH_SIZE
steps_va   = va_g_plain.samples  // BATCH_SIZE
steps_te   = te_g_plain.samples  // BATCH_SIZE + 1

tr_seq_plain = flat_sequence_generator(tr_g_plain)
va_seq_plain = flat_sequence_generator(va_g_plain)
te_seq_plain = flat_sequence_generator(te_g_plain)

hybrid_no_aug = build_improved_hybrid()

t0 = time.time()
hist_no_aug = hybrid_no_aug.fit(
    tr_seq_plain, steps_per_epoch=steps_tr,
    validation_data=va_seq_plain, validation_steps=steps_va,
    epochs=EPOCHS, callbacks=get_callbacks_hybrid(), verbose=1)
epoch_time_h1 = (time.time() - t0) / max(len(hist_no_aug.epoch), 1)

model_histories['Improved Hybrid (No Aug)'] = hist_no_aug.history

te_g_plain.reset()
t_inf = time.time()
y_prob_h1 = hybrid_no_aug.predict(te_seq_plain, steps=steps_te, verbose=0)
inf_ms_h1 = (time.time() - t_inf) / te_g_plain.samples * 1000

timing_results.append({
    'Model': 'Improved Hybrid (No Aug)',
    'Training time (s/epoch)': round(epoch_time_h1, 2),
    'Inference time (ms/image)': round(inf_ms_h1, 2)
})

y_pred_h1 = np.argmax(y_prob_h1, axis=1)
y_true_h1 = te_g_plain.classes[:len(y_pred_h1)]

model_results['Improved Hybrid (No Aug)'] = {
    'Accuracy':  accuracy_score(y_true_h1, y_pred_h1) * 100,
    'Precision': precision_score(y_true_h1, y_pred_h1, average='macro') * 100,
    'Recall':    recall_score(y_true_h1, y_pred_h1, average='macro') * 100,
    'F1-Score':  f1_score(y_true_h1, y_pred_h1, average='macro') * 100,
}
model_preds['Improved Hybrid (No Aug)'] = {
    'y_true': y_true_h1, 'y_prob': y_prob_h1, 'y_pred': y_pred_h1}

print("\nAblasyon sonuclari - Gelistirilmis Hibrit (Artirma Yok)")
for k, v in model_results['Improved Hybrid (No Aug)'].items():
    print(f"  {k}: {v:.2f}%")

In [ ]:
# Onerilen Gelistirilmis Hibrit olarak EfficientNetB0 + Bi-LSTM + Yapilandirilmis Artirma

print("\nOnerilen Gelistirilmis Hibrit egitiliyor")
print("Mimari olarak EfficientNetB0 (dondurulmus) + Cift Yonlu LSTM(128) + Yapilandirilmis Artirma")
tf.keras.backend.clear_session(); gc.collect()

# Dizi olusturma icin temel uretici
tr_g_aug  = make_gen(ImageDataGenerator(rescale=1.0/255), TRAIN_DIR, shuffle=True)
va_g_eval = make_gen(val_test_gen_base, VAL_DIR, shuffle=False)
te_g_eval = make_gen(val_test_gen_base, TEST_DIR, shuffle=False)

steps_tr2 = tr_g_aug.samples  // BATCH_SIZE
steps_va2 = va_g_eval.samples // BATCH_SIZE
steps_te2 = te_g_eval.samples // BATCH_SIZE + 1

# Egitim icin yapilandirilmis kademeli diziler
tr_seq_aug  = structured_sequence_generator(tr_g_aug,  add_noise=True)
# Dogrulama / test icin duz diziler (degerlendirme sirasinda artirma yok)
va_seq_flat = flat_sequence_generator(va_g_eval)
te_seq_flat = flat_sequence_generator(te_g_eval)

hybrid_proposed = build_improved_hybrid()

t0 = time.time()
hist_proposed = hybrid_proposed.fit(
    tr_seq_aug,  steps_per_epoch=steps_tr2,
    validation_data=va_seq_flat, validation_steps=steps_va2,
    epochs=EPOCHS, callbacks=get_callbacks_hybrid(), verbose=1)
epoch_time_h2 = (time.time() - t0) / max(len(hist_proposed.epoch), 1)

model_histories['Improved Hybrid (Proposed)'] = hist_proposed.history

te_g_eval.reset()
t_inf = time.time()
y_prob_h2 = hybrid_proposed.predict(te_seq_flat, steps=steps_te2, verbose=0)
inf_ms_h2 = (time.time() - t_inf) / te_g_eval.samples * 1000

timing_results.append({
    'Model': 'Improved Hybrid (Proposed)',
    'Training time (s/epoch)': round(epoch_time_h2, 2),
    'Inference time (ms/image)': round(inf_ms_h2, 2)
})

y_pred_h2 = np.argmax(y_prob_h2, axis=1)
y_true_h2 = te_g_eval.classes[:len(y_pred_h2)]

model_results['Improved Hybrid (Proposed)'] = {
    'Accuracy':  accuracy_score(y_true_h2, y_pred_h2) * 100,
    'Precision': precision_score(y_true_h2, y_pred_h2, average='macro') * 100,
    'Recall':    recall_score(y_true_h2, y_pred_h2, average='macro') * 100,
    'F1-Score':  f1_score(y_true_h2, y_pred_h2, average='macro') * 100,
}
model_preds['Improved Hybrid (Proposed)'] = {
    'y_true': y_true_h2, 'y_prob': y_prob_h2, 'y_pred': y_pred_h2}

print("\n" + "="*60)
print("Onerilen Gelistirilmis Hibrit Sonuclari")
for k, v in model_results['Improved Hybrid (Proposed)'].items():
    print(f"  {k}: {v:.2f}%")

# BOLUM 5 - Sonuclar, Tablolar ve Gorsellestirmeler

In [ ]:
# Performans Karsilastirma Tablosu
df_all = pd.DataFrame(model_results).T.round(2)
df_timing = pd.DataFrame(timing_results).set_index('Model')

print("\n Model Performans Karsilastirmasi")
display(df_all)
print("\n Egitim ve Cikarim (Inference) Sureleri")
display(df_timing)

# Orijinal makaleden referans degerleri
paper_ref = {
    'InceptionV3':   {'Accuracy': 89.2, 'F1-Score': 88.1, 'AUC': 0.93},
    'VGG16':         {'Accuracy': 85.6, 'F1-Score': 84.2, 'AUC': 0.89},
    'ResNet50':      {'Accuracy': 91.1, 'F1-Score': 90.0, 'AUC': 0.95},
    'DenseNet121':   {'Accuracy': 91.6, 'F1-Score': 90.5, 'AUC': 0.96},
    'EfficientNetB0':{'Accuracy': 92.4, 'F1-Score': 91.4, 'AUC': 0.97},
    'Hybrid CNN-LSTM (makale)': {'Accuracy': 98.9, 'F1-Score': 97.8, 'AUC': 0.99},
}
df_paper = pd.DataFrame(paper_ref).T
print("\n Ghosh & Singh (2026)'den Referans Degerleri")
display(df_paper)

In [ ]:
# Dogruluk karsilastirmasi
models = list(df_all.index)
colors = ['#3498db']*5 + ['#f39c12', '#2ecc71']

plt.figure(figsize=(13, 6))
bars = plt.bar(models, df_all['Accuracy'], color=colors, edgecolor='black', width=0.6)
plt.axhline(y=98.9, color='red', linestyle='--', linewidth=1.5, label='Paper best (98.9%)')
plt.title('Dogruluk Karsilastirmasi - Gelistirilmis Modeller ve Makale Referansi',
          fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=12)
plt.ylim(0, 115)
plt.xticks(rotation=20, ha='right')
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.1f}%',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('fig_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Kesinlik, Duyarlilik, F1 gruplandirilmis cubuk grafigi
metrics = ['Precision', 'Recall', 'F1-Score']
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(16, 6))
metric_colors = ['#2ecc71', '#3498db', '#e74c3c']
for i, (m, c) in enumerate(zip(metrics, metric_colors)):
    ax.bar(x + (i-1)*width, df_all[m], width, label=m, color=c,
           alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha='right')
ax.set_ylim(0, 115)
ax.set_ylabel('Skor (%)', fontsize=12)
ax.set_title('Kesinlik, Duyarlilik ve F1-Skoru - Tum Gelistirilmis Modeller',
             fontsize=14, fontweight='bold')
ax.axhline(y=97.8, color='red', linestyle='--', linewidth=1, label='Paper F1 best (97.8%)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_precision_recall_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tum modeller icin karmasiklik (confusion) matrisleri
n_models = len(models)
ncols = 3
nrows = (n_models + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 6*nrows))
axes = axes.ravel()

for idx, m_name in enumerate(models):
    cm = confusion_matrix(
        model_preds[m_name]['y_true'],
        model_preds[m_name]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(m_name, fontweight='bold', fontsize=10)
    axes[idx].set_xlabel('Tahmin Edilen')
    axes[idx].set_ylabel('Gercek')

for idx in range(n_models, len(axes)):
    axes[idx].axis('off')

plt.suptitle('Karmasiklik Matrisleri - Tum Modeller', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC egrileri (makro-ortalama)
y_true_bin = label_binarize(
    model_preds['Improved Hybrid (Proposed)']['y_true'],
    classes=np.arange(NUM_CLASSES))

plt.figure(figsize=(10, 8))
cmap = plt.cm.get_cmap('tab10')

for idx, m_name in enumerate(models):
    y_prob = model_preds[m_name]['y_prob']
    y_true_local = label_binarize(
        model_preds[m_name]['y_true'], classes=np.arange(NUM_CLASSES))
    fprs, tprs, roc_aucs = {}, {}, {}
    for c in range(NUM_CLASSES):
        fprs[c], tprs[c], _ = roc_curve(y_true_local[:, c], y_prob[:, c])
        roc_aucs[c] = auc(fprs[c], tprs[c])
    all_fpr = np.unique(np.concatenate([fprs[c] for c in range(NUM_CLASSES)]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in range(NUM_CLASSES):
        mean_tpr += np.interp(all_fpr, fprs[c], tprs[c])
    mean_tpr /= NUM_CLASSES
    macro_auc = auc(all_fpr, mean_tpr)
    plt.plot(all_fpr, mean_tpr, color=cmap(idx),
             label=f'{m_name} (AUC={macro_auc:.3f})', lw=1.5)

plt.plot([0,1],[0,1],'k--',lw=0.8)
plt.xlabel('False Positive Orani')
plt.ylabel('True Positive Orani')
plt.title('ROC Egrisi Karsilastirmasi — Makro Ortalama', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Egitim ve dogrulama kaybi / dogrulugu - Onerilen Gelistirilmis Hibrit
h = model_histories['Improved Hybrid (Proposed)']
epochs_x = range(1, len(h['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_x, h['loss'], label='Egitim Kaybi', color='blue')
axes[0].plot(epochs_x, h['val_loss'], label='Dogrulama Kaybi', color='red')
axes[0].set_title('Epochlar Boyunca Kayip - Gelistirilmis Hibrit', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Kayip')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, h['accuracy'], label='Egitim Dogrulugu', color='blue')
axes[1].plot(epochs_x, h['val_accuracy'], label='Dogrulama Dogrulugu', color='red')
axes[1].set_title('Epochlar Boyunca Dogruluk - Gelistirilmis Hibrit', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dogruluk')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Onerilen Gelistirilmis Hibrit icin sinif bazli karmasiklik matrisi
cm_hybrid = confusion_matrix(
    model_preds['Improved Hybrid (Proposed)']['y_true'],
    model_preds['Improved Hybrid (Proposed)']['y_pred'])

plt.figure(figsize=(9, 7))
sns.heatmap(cm_hybrid, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Sinif Bazli Karmasiklik Matrisi — Gelistirilmis Hibrit (Onerilen)',
          fontsize=13, fontweight='bold')
plt.ylabel('Gercek Sinif'); plt.xlabel('Tahmin Edilen Sinif')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('fig_per_class_cm.png', dpi=150, bbox_inches='tight')
plt.show()

# Sinif bazli metrikler
from sklearn.metrics import classification_report
print("\n Sinif Bazli Siniflandirma Raporu — Gelistirilmis Hibrit")
print(classification_report(
    model_preds['Improved Hybrid (Proposed)']['y_true'],
    model_preds['Improved Hybrid (Proposed)']['y_pred'],
    target_names=CLASS_NAMES, digits=4))

In [ ]:
# Ablasyon Calismasi Ozeti
ablation = {
    'Temel Model (Artirma Yok, Ozel CNN)': model_results.get('Hybrid CNN-LSTM (No Aug)',
        {'Accuracy': 97.87, 'Precision': 97.74, 'Recall': 97.80, 'F1-Score': 97.80}),
    'Gelistirilmis (Artirma Yok, EfficientNetB0)': model_results['Improved Hybrid (No Aug)'],
    'Onerilen (EfficientNetB0 + Bi-LSTM + Yapi.Artirma)': model_results['Improved Hybrid (Proposed)'],
    'Makale Hibrit (referans)': {
        'Accuracy': 98.9, 'Precision': 97.5, 'Recall': 98.1, 'F1-Score': 97.8}
}

df_ablation = pd.DataFrame(ablation).T.round(2)
print("\n Ablasyon Calismasi")
display(df_ablation)

In [ ]:
# Onerilen hibrit modeli kaydet
hybrid_proposed.save('improved_hybrid_model.keras')

# Yedeklenecek dosyalarin listesi
files_to_backup = [
    'improved_hybrid_model.keras',
    'fig_accuracy_comparison.png',
    'fig_precision_recall_f1.png',
    'fig_confusion_matrices.png',
    'fig_roc_curves.png',
    'fig_training_curves.png',
    'fig_per_class_cm.png'
]

print("Sonuclar Google Drive'a yedekleniyor")
for file in files_to_backup:
    if os.path.exists(file):
        shutil.copy(file, os.path.join(DRIVE_BACKUP_DIR, file))
        print(f"Yedeklendi {file}")